In [1]:
import sys
sys.path.insert(0, '..')

In [2]:
from src.config import get_spark, BRONZE_PRODUCTS, SILVER_PRODUCTS
from delta.tables import DeltaTable

In [3]:
spark = get_spark()

:: loading settings :: url = jar:file:/usr/local/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-e422bcc5-12fb-44e5-82b0-d8d4a73145d6;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.3.0 in central
	found io.delta#delta-storage;3.3.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 223ms :: artifacts dl 11ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.3.0 from central in [default]
	io.delta#delta-storage;3.3.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   |   0   |  

In [4]:
print('=== BRONZE ===')
df_b = spark.read.format('delta').load(str(BRONZE_PRODUCTS))
df_b.show(5)
df_b.printSchema()
print('Row count:', df_b.count())
print('=== SILVER ===')
df_s = spark.read.format('delta').load(str(SILVER_PRODUCTS))
df_s.show(5)
df_s.printSchema()
print('Row count:', df_s.count())

=== BRONZE ===


26/06/25 15:24:27 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

+------------+---------------+--------------------+--------------+--------------------+--------------------+-------------+----------------+--------------------+--------------------+----------------+--------+------------------+------------------+-----------+-------------+---------+--------------------+
|        code|last_modified_t|        product_name|        brands|       categories_en|           labels_en| countries_en|nutriscore_grade|    ingredients_tags|      food_groups_en|energy-kcal_100g|fat_100g|saturated-fat_100g|carbohydrates_100g|sugars_100g|proteins_100g|salt_100g|         ingest_at_t|
+------------+---------------+--------------------+--------------+--------------------+--------------------+-------------+----------------+--------------------+--------------------+----------------+--------+------------------+------------------+-----------+-------------+---------+--------------------+
|000000000054|     1733085204|Limonade artisana...|          NULL|                NULL|    

+--------+---------------+--------------------+-------------+--------------------+---------+--------------+----------------+--------------------+--------------------+----------------+--------+------------------+------------------+-----------+-------------+---------+
|    code|last_modified_t|        product_name|       brands|       categories_en|labels_en|  countries_en|nutriscore_grade|    ingredients_tags|      food_groups_en|energy-kcal_100g|fat_100g|saturated-fat_100g|carbohydrates_100g|sugars_100g|proteins_100g|salt_100g|
+--------+---------------+--------------------+-------------+--------------------+---------+--------------+----------------+--------------------+--------------------+----------------+--------+------------------+------------------+-----------+-------------+---------+
|00000010|     1750695017|                 xxx|          xxx|Beverages and bev...|   it:xxx|       Germany|            NULL|[wheat-flour, cer...|Beverages,Sweeten...|            NULL|    NULL|       

In [5]:
DeltaTable.forPath(spark, str(BRONZE_PRODUCTS)).history().show(truncate=False)

+-------+-----------------------+------+--------+---------+--------------------------------------+----+--------+---------+-----------+--------------+-------------+----------------------------------------------------------------------+------------+-----------------------------------+
|version|timestamp              |userId|userName|operation|operationParameters                   |job |notebook|clusterId|readVersion|isolationLevel|isBlindAppend|operationMetrics                                                      |userMetadata|engineInfo                         |
+-------+-----------------------+------+--------+---------+--------------------------------------+----+--------+---------+-----------+--------------+-------------+----------------------------------------------------------------------+------------+-----------------------------------+
|0      |2026-06-25 15:18:37.226|NULL  |NULL    |WRITE    |{mode -> Overwrite, partitionBy -> []}|NULL|NULL    |NULL     |NULL       |Serializable  

In [6]:
DeltaTable.forPath(spark, str(SILVER_PRODUCTS)).history().show(truncate=False)

+-------+-----------------------+------+--------+---------+--------------------------------------+----+--------+---------+-----------+--------------+-------------+----------------------------------------------------------------------+------------+-----------------------------------+
|version|timestamp              |userId|userName|operation|operationParameters                   |job |notebook|clusterId|readVersion|isolationLevel|isBlindAppend|operationMetrics                                                      |userMetadata|engineInfo                         |
+-------+-----------------------+------+--------+---------+--------------------------------------+----+--------+---------+-----------+--------------+-------------+----------------------------------------------------------------------+------------+-----------------------------------+
|0      |2026-06-25 15:22:32.398|NULL  |NULL    |WRITE    |{mode -> Overwrite, partitionBy -> []}|NULL|NULL    |NULL     |NULL       |Serializable  

In [7]:
spark.stop()